In [ ]:
import geopandas as gpd
import esda
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from libpysal import graph
data_path = Path("../data")

In [ ]:
simd_2020 = gpd.read_file(f"{data_path}/SIMD_DATA/SG_SIMD_2020.shp").set_index("DataZone")

In [ ]:
simd_2020.head()

In [ ]:
if simd_2020.crs != "EPSG:27700":
    simd_2020 = simd_2020.to_crs(epsg=27700)

simd_2020.describe()

In [ ]:
# Join counts on binary variable Rankv2
simd_2020["rankv2_binary"] = (simd_2020["Rankv2"] < simd_2020["Rankv2"].mean()).astype(int)
simd_2020[["Rankv2", "rankv2_binary"]].sample(8)

In [ ]:
_ = simd_2020.plot("rankv2_binary", cmap="binary", linewidth=.1, edgecolor="grey")

From what I see the join counts don't display much spatial pattern outside of urban areas. Yet in urban areas like the central belt, there might be something to explore further.

In [ ]:
simd_2020.index

In [ ]:
contiguity = graph.Graph.build_contiguity(simd_2020, rook=False)

rank2_jc = esda.Join_Counts(
    simd_2020["rankv2_binary"], contiguity
)

In [ ]:
contiguity["S01013091"]

In [ ]:
print(rank2_jc.bb, rank2_jc.ww, rank2_jc.bw, rank2_jc.J)

In [ ]:
# Calculate spatial lag from row_queen
contiguity_r = contiguity.transform("r")
contiguity_r["S01013091"]

In [ ]:
simd_2020["rank2_lag"] = contiguity_r.lag(simd_2020["Rankv2"])
simd_2020[["rank2_lag", "Rankv2"]]

In [ ]:
simd_2020["rankv2_z"] = (
    simd_2020["Rankv2"] - simd_2020["Rankv2"].mean()
) / simd_2020["Rankv2"].std()

simd_2020["rankv2_z_lag"] = contiguity_r.lag(simd_2020["rankv2_z~"])

In [ ]:
f, ax = plt.subplots(1, figsize=(6, 6))
sns.regplot(
    x="rankv2_z",
    y="rankv2_z_lag",
    data=simd_2020,
    marker=".",
    scatter_kws={"alpha": 0.2},
    line_kws=dict(color="lightcoral")
)
ax.set_aspect('equal')
plt.axvline(0, c="black", alpha=0.5)
plt.axhline(0, c="black", alpha=0.5)
plt.text(1.3, 2.7, "High-high", fontsize=10)
plt.text(1.3, -2.7, "High-low", fontsize=10)
plt.text(-2.4, 2.7, "Low-high", fontsize=10)
plt.text(-2.4, -2.7, "Low-low", fontsize=10);

In [ ]:
# LISA
mi = esda.Moran(simd_2020["Rankv2"], contiguity_r)

In [ ]:
print(f"Global Moran I: {mi.I}")
print(f"Global Moran I (p-value): {mi.p_sim}")

So globally it seems that Moran I is positive, > 5, indicating that there is global spatial autocorrelation (cluster) with p-value <0.05 suggesting that this is statistically significant.

In [ ]:
k = [5, 10, 25, 50, 75, 100, 250, 500, 1000]

correlogram = esda.correlogram(
    geometry=simd_2020.representative_point(),
    variable=simd_2020["Rankv2"],
    support=k,
    distance_type="knn"
)

In [ ]:
ax = correlogram.I.plot()
ax.set_xlabel("KNN")
_ = ax.set_ylabel("Moran's I")

In [ ]:
# LISA
lisa = esda.Moran_Local(simd_2020["Rankv2"], contiguity_r)
simd_2020["cluster"] = lisa.get_cluster_labels(crit_value = 0.05)
simd_2020.head()

In [ ]:
lisa.explore(
  simd_2020,
  crit_value=0.05,
  prefer_canvas=True,
  tiles="CartoDB Positron",
)

In [ ]:
_ = lisa.plot_scatter()